# Bike Share Toronto Ridership datasets (Toronto Open Data)

In [21]:
import pandas as pd 
import json
from pathlib import Path
RAW_DIR = Path("data/raw")

In [6]:
data_2024 = pd.read_csv(RAW_DIR / "bikeshare-ridership-2024.csv")

In [ ]:
# with open (RAW_DIR/"station_information.json") as file:
#     stations = json.load(file)

In [7]:
data_2024.head(5)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
0,26682740,528,7041,2024-01-01 00:00:08,Edward St / Yonge St,7130.0,2024-01-01 00:08:56,Davenport Rd / Bedford Rd,7855,Casual,EFIT G5
1,26682742,1924,7245,2024-01-01 00:02:04,Tecumseth St / Queen St W - SMART,7259.0,2024-01-01 00:34:08,Lower Spadina Ave / Lake Shore Blvd W,2453,Casual,ICONIC
2,26682743,2741,7052,2024-01-01 00:03:25,Wellington St W / Bay St,7366.0,2024-01-01 00:49:06,Fort York Blvd / Bathurst St - SMART,7332,Member,ICONIC
3,26682744,353,7041,2024-01-01 00:03:39,Edward St / Yonge St,7418.0,2024-01-01 00:09:32,College Park - Yonge St Entrance,782,Member,ICONIC
4,26682746,2118,7721,2024-01-01 00:04:44,Widmer St / King St W,7721.0,2024-01-01 00:40:02,Widmer St / King St W,1008,Member,ICONIC


In [8]:
data_2024.shape

(6953094, 11)

In [10]:
data_2024.info()

<class 'pandas.DataFrame'>
RangeIndex: 6953094 entries, 0 to 6953093
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Trip_Id             int64  
 1   Trip_Duration       int64  
 2   Start_Station_Id    int64  
 3   Start_Time          str    
 4   Start_Station_Name  str    
 5   End_Station_Id      float64
 6   End_Time            str    
 7   End_Station_Name    str    
 8   Bike_Id             int64  
 9   User_Type           str    
 10  Bike_Model          str    
dtypes: float64(1), int64(4), str(6)
memory usage: 1.2 GB


## Initial data checks
Couple of things to check:
- convert the columns to appropriate data types
- split the data into monthly datasets
- check if trip_id has any duplicates
- check for all possible values for user_type and bike_model columns
- check for null values

In [13]:
data_2024["User_Type"].unique()

<ArrowStringArray>
['Casual', 'Member']
Length: 2, dtype: str

In [65]:
data_2024["Bike_Id"].nunique()

8085

In [14]:
data_2024["Bike_Model"].unique()

<ArrowStringArray>
['EFIT G5', 'ICONIC', 'EFIT']
Length: 3, dtype: str

In [16]:
data_2024["Trip_Id"].duplicated().sum() #check if there are any duplicates in trip_id col

np.int64(0)

### Missing Values

#### Null counts by column

In [17]:
data_2024.isna().sum()

Trip_Id                  0
Trip_Duration            0
Start_Station_Id         0
Start_Time               0
Start_Station_Name       0
End_Station_Id        8424
End_Time              5822
End_Station_Name      8424
Bike_Id                  0
User_Type                0
Bike_Model               0
dtype: int64

In [18]:
data_2024[data_2024.isna().any(axis=1)].head(5)


,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
20,26682765,4327,7881,2024-01-01 00:14:53,Queens Quay E / Merchants' Wharf,NaN,2024-01-01 01:27:00,NaN,3379,Casual,ICONIC
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC
4689,26688136,0,7359,2024-01-02 10:29:25,Rosedale Park (20 Scholfield Ave),NaN,2024-01-02 10:29:25,NaN,1790,Member,ICONIC
5076,26688576,287,7449,2024-01-02 11:57:13,Cosburn Ave / Cedarvale Ave,NaN,2024-01-02 12:02:00,NaN,2302,Casual,ICONIC


In [42]:
data_2024[data_2024["End_Station_Id"].isna()].head(5)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
20,26682765,4327,7881,2024-01-01 00:14:53,Queens Quay E / Merchants' Wharf,NaN,2024-01-01 01:27:00,NaN,3379,Casual,ICONIC
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC
4689,26688136,0,7359,2024-01-02 10:29:25,Rosedale Park (20 Scholfield Ave),NaN,2024-01-02 10:29:25,NaN,1790,Member,ICONIC
5076,26688576,287,7449,2024-01-02 11:57:13,Cosburn Ave / Cedarvale Ave,NaN,2024-01-02 12:02:00,NaN,2302,Casual,ICONIC


In [26]:
data_2024[data_2024["End_Time"].isna()].head(5)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
65291,26757586,0,7527,2024-01-11 16:56:33,Joseph J Piccininni Community Centre,NaN,NaN,NaN,5688,Casual,ICONIC
122444,26822518,0,7204,2024-01-21 18:17:57,College St / Crawford St,NaN,NaN,NaN,752,Member,ICONIC
157386,26863071,0,7029,2024-01-26 19:00:34,Bay St / Bloor St W (East Side),NaN,NaN,NaN,1297,Casual,ICONIC
188346,26898562,0,7281,2024-01-30 14:56:21,Charles St W / Balmuto St - SMART,NaN,NaN,NaN,2988,Casual,ICONIC
228585,26943404,0,7321,2024-02-03 13:05:25,Front St W / University Ave (2),NaN,NaN,NaN,3331,Casual,ICONIC


We've got **8424** records where the "End_Station_Id" and "End_Station_Name" are missing, and **5822** records where the "End_Time" is missing (this could potentially be the subset of those 8424 records)

#### End_Station_Id and End_Station_Name

In [51]:
#check if missing End_Station_Id always corresponds to a missing End_Station_Name (this prob should)
#check if there are rows where the ID is missing but the Station name is not:
data_2024[(data_2024["End_Station_Id"].isna()) & (data_2024["End_Station_Name"].notna())].shape

(0, 11)

In [30]:
#check for rows where the Id exists but the name is missing:
data_2024[data_2024["End_Station_Id"].notna() & data_2024["End_Station_Name"].isna()].shape

(0, 11)

In [34]:
#or just do:
data_2024[data_2024["End_Station_Id"].isna() & data_2024["End_Station_Name"].isna()].shape


(8424, 11)

In [31]:
data_2024[["End_Station_Id", "End_Station_Name"]].isna().value_counts()

End_Station_Id  End_Station_Name
False           False               6944670
True            True                   8424
Name: count, dtype: int64

#### End_Time vs End station

In [36]:
#now check if records where the end_time is nan is a subset of those 8424 records
data_2024[data_2024["End_Time"].isna() & data_2024["End_Station_Id"].isna()].shape


(3313, 11)

Only **3313** of records where End_Time is null are a subset of records where **both** End_Station_Id and End_Station_Name are null

In [39]:
end_time_nan = data_2024[data_2024["End_Time"].isna() & data_2024["End_Station_Id"].notna()]
end_time_nan.shape

(2509, 11)

In [40]:
end_time_nan.head(5)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
2678398,29756233,0,7190,2024-07-02 16:08:02,St. George St / Hoskin Ave,7176.0,NaN,Bathurst St / Fort York Blvd,6307,Casual,ICONIC
2678821,29756682,0,7409,2024-07-02 16:20:00,Isabella St / Church St,7028.0,NaN,Gould St / Mutual St,37,Member,ICONIC
2679069,29756947,0,7374,2024-07-02 16:27:38,Frederick St / The Esplanade,7028.0,NaN,Gould St / Mutual St,1782,Member,ICONIC
2679086,29756966,0,7502,2024-07-02 16:28:05,University Ave / College St (East),7176.0,NaN,Bathurst St / Fort York Blvd,2920,Member,ICONIC
2679583,29757501,0,7164,2024-07-02 16:40:52,Gould St / Yonge St (TMU),7028.0,NaN,Gould St / Mutual St,6499,Member,ICONIC


So it's not always that when End_Station_Id is null, the end_time is also null. Earlier assumption was wrong.

In [41]:
data_2024[["End_Station_Id", "End_Station_Name", "End_Time"]].isna().value_counts()


End_Station_Id  End_Station_Name  End_Time
False           False             False       6942161
True            True              False          5111
                                  True           3313
False           False             True           2509
Name: count, dtype: int64

### Trip_Duration Checks

In [55]:
data_2024[(data_2024["Trip_Duration"] == 0)].shape

(10873, 11)

In [52]:
data_2024[(data_2024["Trip_Duration"] == 0)].head(10)

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
980,26683830,0,7091,2024-01-01 11:12:50,Donlands Subway Station,7309.0,2024-01-01 11:12:50,Queen St E / Rhodes Ave,7398,Member,EFIT G5
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC
4182,26687522,0,7030,2024-01-02 08:52:47,Bay St / Wellesley St W,7713.0,2024-01-02 08:52:47,Temperance St / Bay St,2036,Member,ICONIC
4689,26688136,0,7359,2024-01-02 10:29:25,Rosedale Park (20 Scholfield Ave),NaN,2024-01-02 10:29:25,NaN,1790,Member,ICONIC
5525,26689137,0,7150,2024-01-02 13:13:08,Dufferin St / Sylvan Av (Dufferin Grove Park),NaN,2024-01-02 13:13:08,NaN,2171,Member,ICONIC
6104,26689767,0,7256,2024-01-02 14:49:50,Vanauley St / Queen St W - SMART,NaN,2024-01-02 14:49:50,NaN,692,Member,ICONIC
6735,26690470,0,7032,2024-01-02 16:16:26,Augusta Ave / Dundas St W,NaN,2024-01-02 16:16:26,NaN,6185,Member,ICONIC
8077,26691892,0,7297,2024-01-02 18:07:28,Havelock St / Dufferin Park,NaN,2024-01-02 18:07:28,NaN,2518,Member,ICONIC
9170,26693071,0,7059,2024-01-02 20:33:18,Front St W / Blue Jays Way,7010.0,2024-01-02 20:33:18,King St W / Spadina Ave,7316,Member,ICONIC


In [54]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())].shape

(5829, 11)

In [56]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())].head()

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
980,26683830,0,7091,2024-01-01 11:12:50,Donlands Subway Station,7309.0,2024-01-01 11:12:50,Queen St E / Rhodes Ave,7398,Member,EFIT G5
4182,26687522,0,7030,2024-01-02 08:52:47,Bay St / Wellesley St W,7713.0,2024-01-02 08:52:47,Temperance St / Bay St,2036,Member,ICONIC
9170,26693071,0,7059,2024-01-02 20:33:18,Front St W / Blue Jays Way,7010.0,2024-01-02 20:33:18,King St W / Spadina Ave,7316,Member,ICONIC
9241,26693146,0,7437,2024-01-02 20:49:28,Concord Ave / Bloor St W,7163.0,2024-01-02 20:49:28,Yonge St / Wood St,8052,Member,EFIT G5
12683,26697235,0,7273,2024-01-03 12:59:46,Bay St / Charles St W - SMART,7022.0,2024-01-03 12:59:46,Simcoe St / Queen St W,5677,Casual,ICONIC


In [66]:
zero_with_end = data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].notna())]
(zero_with_end["Start_Station_Id"] == zero_with_end["End_Station_Id"]).value_counts()

False    5276
True      553
Name: count, dtype: int64

553 -> 0 Time_Duration, and has the same start and end station  
5276 -> 0 Time_Duration, and different start and end station

In [58]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].isna())].shape

(5044, 11)

In [57]:
data_2024[(data_2024["Trip_Duration"] == 0) & (data_2024["End_Station_Id"].isna())].head()

,Trip_Id,Trip_Duration,Start_Station_Id,Start_Time,Start_Station_Name,End_Station_Id,End_Time,End_Station_Name,Bike_Id,User_Type,Bike_Model
683,26683508,0,7534,2024-01-01 05:46:56,Walnut Ave / Queen St W,NaN,2024-01-01 05:46:56,NaN,6386,Member,ICONIC
2642,26685566,0,7437,2024-01-01 19:08:20,Concord Ave / Bloor St W,NaN,2024-01-01 19:08:20,NaN,6409,Member,ICONIC
4689,26688136,0,7359,2024-01-02 10:29:25,Rosedale Park (20 Scholfield Ave),NaN,2024-01-02 10:29:25,NaN,1790,Member,ICONIC
5525,26689137,0,7150,2024-01-02 13:13:08,Dufferin St / Sylvan Av (Dufferin Grove Park),NaN,2024-01-02 13:13:08,NaN,2171,Member,ICONIC
6104,26689767,0,7256,2024-01-02 14:49:50,Vanauley St / Queen St W - SMART,NaN,2024-01-02 14:49:50,NaN,692,Member,ICONIC


In [59]:
data_2024[(data_2024["Trip_Duration"] == 0) & ((data_2024["Start_Station_Id"].isna()) & (data_2024["End_Station_Id"].isna()))].shape

(0, 11)

10,873 trips have a Trip_Duration of 0 and Start_Time and End_Time equals for all of them. 3 groups:
- 5044: have no end station. These look like rentals that never became real trips
- 553: start and end at the same station. looks like bike being undocked and returned right away
- 5276: start and end at different stations, but a trip between two stationca cant take zero seconds => timestamps wrong? 

all three groups will be excluded from 'trips'